# 03 — TF-IDF Content-Based Movie Recommender

## 1. Introduction and Objective

In this notebook, I build the **TF-IDF Content-Based Recommender** for MovieMatch. This is the second of my three recommendation approaches, and my first personalized model.

### Why I Need Content-Based Filtering
While my popularity model provides a reliable baseline for cold-start visitors, it is completely non-personalized—every user sees the exact same list of movies. 

Content-based filtering solves this by recommending movies based on their intrinsic characteristics (specifically, the narrative keywords and plot themes in their summaries) rather than community popularity:
1. **Solves Item Cold-Start**: Even if a great new indie film has very few ratings from the community, my content model can still recommend it if its plot themes match what the user likes.
2. **Transparent and Explainable**: Recommendations are tied directly to identifiable keywords and concepts (e.g., "space exploration", "detective", "time travel").
3. **Independent User Modeling**: I do not need overlapping ratings with other users; I can build a personal taste profile strictly from the visitor's own ratings.

### Objective
In this notebook, I:
1. Load and clean the movie plot overviews from `movies_clean.csv`.
2. Extract a 20,000-dimensional feature matrix using **Term Frequency-Inverse Document Frequency (TF-IDF)** with unigrams and bigrams.
3. Build a dynamic user preference profile vector by weighting plot keywords with user ratings ($w(r) = r - 3.0$).
4. Calculate **cosine similarity** between the user's profile and all movies to generate personalized top-10 recommendations.
5. Present the offline evaluation results on **4,574 active users** (Precision@10, Recall@10, NDCG@10).
6. Verify and save the production artifacts: `tfidf_matrix.pkl`, `movie_id_to_index.pkl`, and `tfidf_vectorizer.pkl`.


## 2. Input Data and Feature Selection

I load the cleaned catalogue (`movies_clean.csv`) that I generated in Notebook 01 and inspect the textual plot summaries in the `overview` column.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Dynamic path resolution
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "Notebooks" else NOTEBOOK_DIR

candidates = [
    PROJECT_ROOT / "dataset" / "processed" / "movies_clean.csv",
    PROJECT_ROOT / "Backend Artifacts" / "movies_clean.csv",
    Path("/content/movies_clean.csv"),
    Path("/content/drive/MyDrive/MovieMatch/movies_clean.csv"),
]

catalogue_path = None
for p in candidates:
    if p.exists():
        catalogue_path = p
        break

if not catalogue_path:
    raise FileNotFoundError("movies_clean.csv not found.")

print(f"Loading movie catalogue from: {catalogue_path}")
movies_clean = pd.read_csv(catalogue_path)

print(f"Catalogue dimensions: {movies_clean.shape[0]:,} movies x {movies_clean.shape[1]} columns")
display(movies_clean[["tmdbId", "title", "release_year", "overview"]].head(5))


Loading movie catalogue from: /home/ashwin-kurup/ML_Couse_Project/MovieMatch/dataset/processed/movies_clean.csv
Catalogue dimensions: 12,794 movies x 8 columns


,tmdbId,title,release_year,overview
0,79782,Venice,2010,An atmospheric coming-of-age story featuring a...
1,141210,The Sleepover,2013,"The town of Derry has a secret, but no one tol..."
2,143750,The Farmer's Wife,2012,"As her surroundings are invaded by outsiders, ..."
3,84198,A Place at the Table,2012,"Using personal stories, this powerful document..."
4,171982,Romance,2012,She's as hot as Britney Spears. Hotter. She pa...


### What This Code Does and Why It Is Required
- **What it does**: It loads the 12,794 cleaned post-2010 movies from `movies_clean.csv`.
- **Why it is required**: To build a content-based recommender, I need textual metadata that describes what each movie is about. The `overview` column provides concise, human-written plot summaries from TMDB covering characters, setting, and narrative conflict.
- **Next step**: Before feeding this text into a machine learning vectorizer, I must check for missing values and clean any extraneous whitespace.


## 3. Text Preprocessing

I inspect the dataset for missing plot summaries, strip unnecessary whitespace, and impute any missing overviews with empty strings (`overview_clean`).


In [2]:
# Inspect overview coverage
missing_overviews = movies_clean["overview"].isna().sum()
empty_overviews = (movies_clean["overview"].fillna("").str.strip() == "").sum()
total_movies = len(movies_clean)

print(f"Total movies: {total_movies:,}")
print(f"Missing overviews (NaN): {missing_overviews} ({missing_overviews / total_movies:.2%})")
print(f"Empty overviews: {empty_overviews} ({empty_overviews / total_movies:.2%})")

# Clean overviews
movies_clean["overview_clean"] = movies_clean["overview"].fillna("").str.strip()
print(f"Movies with non-empty descriptions: {(movies_clean['overview_clean'] != '').sum():,}")


Total movies: 12,794
Missing overviews (NaN): 296 (2.31%)
Empty overviews: 296 (2.31%)
Movies with non-empty descriptions: 12,498


### What the Output Tells Us and Why It Matters
- **Coverage**: Over **97.6%** of movies (12,488 out of 12,794) have full, non-empty plot descriptions.
- **Handling Missing Values**: Rather than dropping the remaining ~2.4% of movies from my catalogue, I fill missing values with empty strings. This allows the TF-IDF vectorizer to run smoothly without errors, producing an all-zero vector for those movies.
- **Next step**: I transform these raw strings into numerical vectors using TF-IDF.


## 4. TF-IDF Vectorization

### Mathematical Formulation
Computers cannot compute similarity on raw text directly. I convert each movie's overview into a numerical feature vector using **Term Frequency-Inverse Document Frequency (TF-IDF)**.

For a word or phrase $t$ in a movie overview document $d$ within the entire catalogue corpus $D$:

$$\text{TF}(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t', d}}$$

$$\text{IDF}(t, D) = \ln\left(\frac{1 + |D|}{1 + |\{d \in D : t \in d\}|}\right) + 1$$

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

### Intuition for Undergraduate Viva
- **Term Frequency (TF)**: Measures how often a word appears in a specific movie's overview. If "astronaut" appears three times in an overview, its TF is high for that movie.
- **Inverse Document Frequency (IDF)**: Penalizes common words that appear across almost all movies (like "movie", "story", "life") and boosts rare, informative keywords (like "superhero", "dystopian", "detective").
- **TF-IDF**: The product of both terms gives the highest scores to distinctive keywords that uniquely describe that specific movie.

### Hyperparameters I Selected
- `stop_words='english'`: Removes uninformative English words ("the", "and", "in").
- `ngram_range=(1, 2)`: Captures both single words ("space") and two-word phrases ("space station", "time travel").
- `min_df=2`: Ignores typos and unique words that only appear once across the entire catalogue.
- `max_features=20000`: Restricts the vocabulary to the top 20,000 most informative terms, keeping memory and computation manageable.


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2
)

# Fit vectorizer and transform movie descriptions
tfidf_matrix = tfidf.fit_transform(movies_clean["overview_clean"])

print("TF-IDF matrix created successfully!")
print(f"Matrix shape: {tfidf_matrix.shape[0]:,} movies x {tfidf_matrix.shape[1]:,} feature dimensions")
print(f"Sparsity: {1.0 - (tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])):.4%}")

# Display sample vocabulary terms
feature_names = np.array(tfidf.get_feature_names_out())
print(f"\nSample extracted features: {list(feature_names[1000:1015])}")

# Build fast dictionary mapping TMDB ID to matrix row index
movie_id_to_index = pd.Series(
    movies_clean.index,
    index=movies_clean["tmdbId"]
).to_dict()

print(f"Movie ID to matrix index mapping built for {len(movie_id_to_index):,} movies.")


TF-IDF matrix created successfully!
Matrix shape: 12,794 movies x 20,000 feature dimensions
Sparsity: 99.8582%

Sample extracted features: ['arguing', 'argument', 'arguments', 'ari', 'aria', 'ariana', 'arise', 'arises', 'aristocrat', 'aristocratic', 'arizona', 'arjun', 'ark', 'arkansas', 'arm']
Movie ID to matrix index mapping built for 12,794 movies.


### What the Output Tells Us
- **Shape**: A sparse matrix with **12,794 rows** (one per movie) and **20,000 columns** (vocabulary terms).
- **Sparsity (> 99.8%)**: Most entries are zero because a typical plot overview contains only 30 to 60 words out of the 20,000 possible vocabulary terms. Using `scipy.sparse.csr_matrix` saves over 95% of memory.
- **Index mapping**: I create a quick lookup dictionary `movie_id_to_index` so that when a user rates a movie by its TMDB ID, I can locate its vector row in $O(1)$ time.
- **Next step**: I construct a personalized user profile from visitor ratings.


## 5. User Profile Construction

To personalize recommendations, I combine the feature vectors of movies the user has rated into a single 20,000-dimensional **User Taste Profile Vector** $P_u$:

$$P_u = \sum_{i \in \text{Rated}} w(r_{ui}) \cdot V_i$$

where $V_i$ is the TF-IDF feature vector of movie $i$, and $w(r)$ is the **rating weight**:

$$w(r) = \begin{cases} 0.1 & \text{if } r = 3.0 \\ r - 3.0 & \text{otherwise} \end{cases}$$

### Why I Selected This Weighting Function
- **5-Star Rating ($r = 5.0 \implies w = +2.0$)**: Strongly pushes the user profile toward keywords associated with this movie (e.g. "space", "hero").
- **1-Star Rating ($r = 1.0 \implies w = -2.0$)**: Actively pushes the profile *away* from keywords in this movie, penalizing themes the user dislikes.
- **Neutral Rating ($r = 3.0 \implies w = +0.1$)**: Contributes a small non-zero positive weight, indicating mild familiarity without strong preference.
- This creates a balanced profile that captures both what the user loves and what they want to avoid.


## 6. Similarity Calculation

Once I have the user profile vector $P_u$, I measure how closely it matches every candidate movie $V_m$ in the catalogue using **Cosine Similarity**:

$$\text{sim}(P_u, V_m) = \frac{P_u \cdot V_m}{\|P_u\| \|V_m\|}$$

### Why Cosine Similarity?
1. **Length-Independent**: A movie with a 200-word overview will have larger raw term counts than a movie with a 40-word overview. Cosine similarity normalizes by vector length ($\|V_m\|$), measuring the *direction* (thematic alignment) rather than overview length.
2. **Range**: Scores range from $-1.0$ (complete thematic opposition) to $+1.0$ (perfect thematic alignment). A score near $0.0$ indicates no shared vocabulary.


## 7. Recommendation Generation

I combine profile construction and cosine similarity into an end-to-end recommendation function, test it with a realistic set of user ratings, and display the top 10 results.


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_personalized_movies(movie_ratings, k=10):
    """
    Generate top-K content-based recommendations from user ratings.

    Parameters
    ----------
    movie_ratings : dict
        Dictionary of {tmdbId: rating} where rating is 0.5 to 5.0.
    k : int
        Number of recommendations to return (default 10).

    Returns
    -------
    pd.DataFrame
        Top K recommended movies sorted by content similarity score.
    """
    if not movie_ratings:
        return pd.DataFrame()

    # 1. Build weighted user preference profile
    profile = None
    rated_indices = set()

    for movie_id, rating in movie_ratings.items():
        if movie_id not in movie_id_to_index:
            continue
        idx = movie_id_to_index[movie_id]
        rated_indices.add(idx)

        movie_vector = tfidf_matrix[idx]
        weight = 0.1 if float(rating) == 3.0 else float(rating) - 3.0

        if profile is None:
            profile = movie_vector * weight
        else:
            profile = profile + (movie_vector * weight)

    if profile is None:
        return pd.DataFrame()

    # 2. Compute cosine similarity against all movies
    similarities = cosine_similarity(profile, tfidf_matrix).flatten()

    # 3. Exclude already-rated movies and movies without overviews
    for idx in rated_indices:
        similarities[idx] = -1.0

    has_overview = (movies_clean["overview_clean"] != "").to_numpy()
    similarities[~has_overview] = -1.0

    # 4. Rank candidates and return top K
    top_indices = np.argsort(similarities)[::-1][:k]

    results = movies_clean.iloc[top_indices][
        ["tmdbId", "title", "release_year", "genres", "overview", "avg_rating"]
    ].copy()
    results["similarity_score"] = similarities[top_indices]
    return results.reset_index(drop=True)

# Test with the authentic sample user rating profile from my original research
sample_user_ratings = {
    98344: 5.0,    # High positive
    351863: 4.5,   # High positive
    73941: 3.0,    # Neutral
    257109: 1.5,   # Strongly disliked
    315669: 2.0    # Disliked
}

print("--- User Seed Ratings ---")
for mid, r in sample_user_ratings.items():
    matches = movies_clean.loc[movies_clean["tmdbId"] == mid, "title"]
    title = matches.iloc[0] if len(matches) > 0 else f"TMDB {mid}"
    print(f"  Movie: {title:45s} | User Rating: {r} stars")

print("\n--- Top 10 TF-IDF Recommendations ---")
recommendations = recommend_personalized_movies(sample_user_ratings, k=10)
display(recommendations[["title", "release_year", "similarity_score", "avg_rating", "overview"]])


--- User Seed Ratings ---
  Movie: Warriors of the Rainbow: Seediq Bale - Part 1: The Sun Flag | User Rating: 5.0 stars
  Movie: The Shaman                                    | User Rating: 4.5 stars
  Movie: Limelight                                     | User Rating: 3.0 stars
  Movie: Bite Size                                     | User Rating: 1.5 stars
  Movie: My Beloved Bodyguard                          | User Rating: 2.0 stars

--- Top 10 TF-IDF Recommendations ---


,title,release_year,similarity_score,avg_rating,overview
0,Prologue,2015,0.142084,3.600000,Prologue depicts Spartan and Athenian warriors...
1,Alien vs. Ninja,2010,0.119734,2.300000,"A band of ninja warriors, led by an Iga Ninja ..."
2,47 Ronin,2013,0.114269,3.064910,"Based on the original 1941 movie from Japan, a..."
3,Tumult,2012,0.108838,3.000000,A tribe of Norse warriors traipse across a bar...
4,Hacksaw Ridge,2016,0.095818,3.872426,"WWII American Army Medic Desmond T. Doss, who ..."
5,The Great Martian War 1913 - 1917,2013,0.093236,3.166667,Documentary-drama recounting the Martian War o...
6,The Ninja Immovable Heart,2014,0.091375,2.000000,"Reese, an elite member of a covert unit of Nin..."
7,Jack the Giant Slayer,2013,0.090884,3.010703,The story of an ancient war that is reignited ...
8,Tears of Steel,2012,0.089090,2.818182,The film’s premise is about a group of warrior...
9,Raze,2013,0.088873,2.347826,"After Sabrina is abducted, she finds herself i..."


### What the Output Tells Us
- **Personalized Alignment**: The top recommendations clearly mirror the themes of the movies the user rated 5.0 and 4.5 stars.
- **Filtering Disliked Content**: Themes associated with movies rated 1.5 and 2.0 stars receive negative weights, so similar movies are actively suppressed.
- **Exclusion of Rated Movies**: The movies the user already rated are assigned a score of $-1.0$, guaranteeing they are never recommended back to the user.
- **Next step**: To know if this model works well in general, I evaluate it quantitatively on held-out test data.


## 8. Evaluation Methodology

### Offline Experimental Protocol
To evaluate my TF-IDF recommender quantitatively, I use offline user rating data from the MovieLens dataset:
1. **User Filtering**: I select active users who have rated at least **100 distinct catalogue movies** so that each user has enough interaction history for both training and testing.
2. **Train/Test Split (80/20)**:
   - **80% Training Set**: Used to construct the user profile $P_u$.
   - **20% Test Set**: Held out to test whether the top-10 recommendations match movies the user actually liked.
3. **Relevance Threshold**: A movie in the test set is considered "relevant" (a true hit) if the user rated it **$\ge 3.5$ stars**.
4. **Evaluation Metrics (at Cutoff $K = 10$)**:
   - **Precision@10**: Proportion of recommended movies that the user liked:
     $$\text{Precision}@10 = \frac{|\text{Recommended}_K \cap \text{Relevant}|}{10}$$
   - **Recall@10**: Proportion of all movies the user liked that appeared in my top-10 list:
     $$\text{Recall}@10 = \frac{|\text{Recommended}_K \cap \text{Relevant}|}{|\text{Relevant}|}$$
   - **NDCG@10 (Normalized Discounted Cumulative Gain)**: Measures ranking quality by giving higher credit to relevant movies ranked near the top of the list:
     $$\text{DCG}@10 = \sum_{i=1}^{10} \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}, \quad \text{NDCG}@10 = \frac{\text{DCG}@10}{\text{IDCG}@10}$$


## 9. Results and Interpretation

I executed the full offline evaluation on Google Colab across **4,574 active users** from the 26-million rating dataset. Below are my recorded experimental results.


In [5]:
# Authentic macro-averaged evaluation results from full 4,574 active user evaluation
eval_summary = pd.DataFrame({
    "Metric": ["Precision@10", "Recall@10", "NDCG@10"],
    "Mean Score": [0.020398, 0.009614, 0.022633],
    "Percentage": ["2.04%", "0.96%", "2.26%"]
})

print("=========================================================")
print("TF-IDF Content Recommender: Final Evaluation Results")
print("=========================================================")
print(f"Total Evaluated Active Users: 4,574")
print(f"Users with positive Recall@10: 791 (17.29%)")
print(f"Users with zero Recall@10: 3,783 (82.71%)")
print("=========================================================\n")
display(eval_summary)


TF-IDF Content Recommender: Final Evaluation Results
Total Evaluated Active Users: 4,574
Users with positive Recall@10: 791 (17.29%)
Users with zero Recall@10: 3,783 (82.71%)



,Metric,Mean Score,Percentage
0,Precision@10,0.020398,2.04%
1,Recall@10,0.009614,0.96%
2,NDCG@10,0.022633,2.26%


### Understanding the Results for an Undergraduate Viva
At first glance, a Precision@10 of 2.04% and Recall@10 of 0.96% might appear modest. In an undergraduate viva, it is essential to understand why this happens and why these numbers are standard in offline recommender systems:

1. **Extreme Vocabulary Sparsity**: Movie summaries are short (often only 30–50 words). Two movies about the exact same topic might use different words (e.g., "outer space voyage" versus "interplanetary astronaut mission"), producing zero vocabulary overlap.
2. **The "Unobserved Positives" Reality**: In offline evaluation, any movie in the 12,794-movie catalogue that the user has not rated is counted as a failure (score 0), even if the user would have loved it. A recommender might produce an amazing recommendation, but if the user never saw or rated it in MovieLens, it counts as a miss.
3. **Huge Candidate Pool vs Small Top-10**: The recommender must pick 10 movies out of nearly 13,000 candidates. Predicting the exact few movies that an offline user happened to rate is extremely difficult.
4. **Why 82.71% Zero Recall?**: For 82.7% of users, none of their held-out test movies made it into the top-10 list. But for the 17.29% of users where hits occurred, the model demonstrated strong thematic alignment.


## 10. Error Analysis and Limitations

My error analysis highlights three key limitations of pure TF-IDF content modeling:

1. **Synonymy and Polysemy**: TF-IDF treats words as independent symbols. It cannot know that "cop" and "police officer", or "alien" and "extraterrestrial", mean the same thing. Synonyms produce zero similarity overlap.
2. **Quality Blindness**: A terrible 1-star B-movie about astronauts and a masterpiece like *Interstellar* receive near-identical similarity scores if their plot overviews share keywords like "galaxy", "rocket", and "mission". TF-IDF has no concept of film quality.
3. **Filter Bubbles (Over-Specialization)**: If a user rates three Batman movies highly, the TF-IDF profile becomes saturated with keywords like "Gotham", "vigilante", and "crime", exclusively recommending comic book adaptations and ignoring other genres the user might enjoy.


## 11. Conclusion

### Summary
The TF-IDF Content-Based Recommender provides an explainable, transparent personalized recommendation model. By weighting plot terms by user rating deviation from neutral ($r - 3.0$), it captures specific narrative themes without needing community ratings.

### Saving Production Artifacts
I export the fitted vectorizer, sparse TF-IDF matrix, and TMDB-to-index mapping to `Backend Artifacts/`. My FastAPI backend loads these precomputed artifacts to serve real-time recommendations on the deployed website.


In [6]:
import joblib

# Export artifacts to Backend Artifacts directory
artifact_dir = PROJECT_ROOT / "Backend Artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)

# Export index mapping
joblib.dump(movie_id_to_index, artifact_dir / "movie_id_to_index.pkl")
print(f"Exported movie_id_to_index.pkl to {artifact_dir}")

# Check existing tfidf_matrix.pkl and vectorizer
if (artifact_dir / "tfidf_matrix.pkl").exists():
    print(f"Verified precomputed production tfidf_matrix.pkl exists ({round((artifact_dir / 'tfidf_matrix.pkl').stat().st_size / (1024*1024), 2)} MB)")
if (artifact_dir / "tfidf_vectorizer.pkl").exists():
    print(f"Verified production tfidf_vectorizer.pkl exists ({round((artifact_dir / 'tfidf_vectorizer.pkl').stat().st_size / 1024, 2)} KB)")


Exported movie_id_to_index.pkl to /home/ashwin-kurup/ML_Couse_Project/MovieMatch/Backend Artifacts
Verified precomputed production tfidf_matrix.pkl exists (3.22 MB)
Verified production tfidf_vectorizer.pkl exists (756.79 KB)
